In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Model} = M(W, b) = Wx+b $$
$$ \frac{\partial M}{\partial W} = X $$
$$ \frac{\partial M}{\partial b} = 1 $$

$$ \diamond \diamond \diamond $$


$$ \text{Probability} = p(z) = sigmoid(z) = \frac{e^z}{e^z+1} $$
$$ \frac{\partial p}{\partial z} = \frac{e^z}{(e^z+1)^2} = p(1-p) $$

$$ \diamond \diamond \diamond $$

$$ \text{Loss} = L(p) = -(y\ln(p)+(1-y)\ln(1-p)) $$
$$ \frac{\partial L}{\partial p} = -\Big(y \frac{1}{p} + (1-y) \frac{1}{1-p}(-1) \Big) = \frac{p-y}{p(1-p)} $$
$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial z} \frac{\partial z}{\partial W} = \frac{p-y}{p(1-p)} \, p(1-p) \, X = (p-y)X $$
$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial z} \frac{\partial z}{\partial b} = \frac{p-y}{p(1-p)} \, p(1-p) \, 1 = (p-y)1 $$

$$ \diamond \diamond \diamond $$

In [ ]:
from torch import exp, rand, Tensor

import import_ipynb
from common import assert_eq, assert_ge, assert_lt, Patient, T # type: ignore

# Convert Patient from 3-classes {healthy, viral, bacterial} to 2-classes {healthy, sick}
def Patient2C(sick: float) -> Patient:
    return Patient([1 - sick, sick, 0.00])


def _linear(X: Tensor, W: Tensor, b: Tensor) -> Tensor:
    return X @ W.T + b


def _model(X: Tensor, W: Tensor, b: Tensor) -> Tensor:
    return _linear(X, W, b)


def _sigmoid(Z: Tensor) -> Tensor:
    return exp(Z) / (exp(Z) + 1)


def _binaryCrossEntropy(P: Tensor, Y: Tensor) -> Tensor:
    return -((Y * P.log()) + ((1 - Y) * (1 - P).log())).mean()


def _loss(P: Tensor, Y: Tensor) -> Tensor:  
    return _binaryCrossEntropy(P, Y)


def logistic_regression_2c_sgd_gradient(X: Tensor, Y: Tensor, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with manual gradient calculation..

    Args:
        X: Input features of shape (Samples, Features)
        y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """

    (s, f) = X.shape

    W = weights = rand(1, f)
    assert_eq(W.shape, (1, f))

    b = bias = rand(1)
    assert_eq(b.shape, (1,))

    for _ in range(epochs):
        z = logits = _model(X, W, b)
        assert_eq(z.shape, (s, 1))

        P = probabilities = _sigmoid(z)
        assert_eq(P.shape, (s, 1))

        dL_dW = (P - Y).T @ X
        assert_eq(dL_dW.shape, (1, f))

        #
        # When reducing the loss function, using `mean()` is more appropriate than `sum()` because 
        # it normalizes the loss by the number of samples, providing a more stable and comparable 
        # loss value across different batch sizes.
        #
        dL_db = (P - Y).mean()
        assert_eq(dL_db.shape, ())

        W = W - lr * dL_dW
        b = b - lr * dL_db
        
        #
        # In the autograd version, computing the loss is required because it serves 
        # as the root of the computational graph for backpropagation. 
        #
        # In the manual‑gradient version, the loss value is not needed for the weight update itself,
        # it is computed only to monitor training progress.
        #

        L = loss = _loss(P, Y)

    return (L.item(), lambda X: _sigmoid(_model(X, W, b)))


def _test_logistic_regression_2c_sgd_gradient(epochs=2000, lr=0.1) -> None:
    training_data = T([Patient2C(0.5).data for _ in range(80)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling
    y = training_data[:, -1].unsqueeze(1)

    (_, model) = logistic_regression_2c_sgd_gradient(X, y, epochs, lr)

    for d in T([Patient2C(1.0).data for _ in range(10)]):
        d[0] /= 100 # The same data scaling as during training.
        d[5] /= 200 # The same data scaling as during training.
        assert_ge(model(d[:-1]), 0.5)
        
    for d in T([Patient2C(0.0).data for _ in range(10)]):
        d[0] /= 100 # The same data scaling as during training.
        d[5] /= 200 # The same data scaling as during training.
        assert_lt(model(d[:-1]), 0.5)


if __name__ == "__main__":
    _test_logistic_regression_2c_sgd_gradient()